## Import

In [5]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import matplotlib.pyplot as plt


device = 'cuda:7'

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data generation

In [2]:
from datasets import NonlinearGaussian

n, d = 10000, 64                                    # < higher d, higher MI
true_rho = 0.7                                      # < higher rho, higher MI
case = '3a'                                         # < choose between ['1a', '1b', '2', '3a', '3b', '3c']


dataset = NonlinearGaussian(n_samples=n, n_dims=d, rho=true_rho, mu=0, case=case)
X0, Y0 = dataset.sample_data(n_samples = n)
X, Y = dataset.transformation(X0, Y0)
MI = dataset.true_mutual_info()                     # < we know GT MI

X, Y = X.to(device), Y.to(device)


print('X size=', X.size(), 'Y size=', Y.size())
print("True MI is", MI)

X size= torch.Size([10000, 32]) Y size= torch.Size([10000, 32])
True MI is 10.773512852220248


## MI estimation

In [3]:
class Hyperparams(object):
    def __init__(self): 
        self.lr = 5e-4
        self.bs = 500
        self.wd = 1e-5
        self.max_iteration = 1250
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]

In [10]:
## Mutual information neural diffusion estimate (MINDE)
from estimators import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

use ema: True bs: 500
finished: t= 0 loss= 1.4978152513504028 loss val= 1.5029184818267822 best val loss= 1.5029184818267822 best t= 0
finished: t= 63 loss= 0.37426936626434326 loss val= 0.32848453521728516 best val loss= 0.3236032724380493 best t= 54
finished: t= 126 loss= 0.31149056553840637 loss val= 0.3318749666213989 best val loss= 0.3162815570831299 best t= 120
finished: t= 189 loss= 0.34524059295654297 loss val= 0.3171806335449219 best val loss= 0.3069606423377991 best t= 140
finished: t= 252 loss= 0.29726293683052063 loss val= 0.3306025266647339 best val loss= 0.3069606423377991 best t= 140
finished: t= 315 loss= 0.3081776797771454 loss val= 0.31382906436920166 best val loss= 0.29302120208740234 best t= 299
finished: t= 378 loss= 0.30705657601356506 loss val= 0.3012200891971588 best val loss= 0.2879784107208252 best t= 361
finished: t= 441 loss= 0.3352733254432678 loss val= 0.33013641834259033 best val loss= 0.2879784107208252 best t= 361
finished: t= 504 loss= 0.29801863431930

In [5]:
## Mutual information neural estimate (MINE)
from estimators import MINE

estimator = MINE(architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)


print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.0015621073544025421 loss val= 0.0018369965255260468 best val loss= 0.0018369965255260468 best t= 0


finished: t= 63 loss= -1.5973963737487793 loss val= -1.5545110702514648 best val loss= -1.5545110702514648 best t= 63


finished: t= 126 loss= -2.2091331481933594 loss val= -1.914445161819458 best val loss= -1.9536197185516357 best t= 121


finished: t= 189 loss= -1.9865820407867432 loss val= -2.043680429458618 best val loss= -2.2610220909118652 best t= 187


finished: t= 252 loss= -2.6496901512145996 loss val= -2.5432448387145996 best val loss= -2.5432448387145996 best t= 252


...

finished: t= 1071 loss= -3.7440812587738037 loss val= -3.0080082416534424 best val loss= -3.1888322830200195 best t= 1033


finished: t= 1134 loss= -3.654998302459717 loss val= -2.932642936706543 best val loss= -3.2553369998931885 best t= 1131


finished: t= 1197 loss= -3.8888654708862305 loss val= -3.0901689529418945 best val loss= -3.2553369998931885 best t= 1131




true MI: 10.773512852220248


est MI: 4.001711845397949


In [7]:
## Neural adaptive MI estimate
from estimators import VCE

estimator = VCE(hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 1.059440016746521 loss val= 1.0527572631835938 best val loss= 1.0527572631835938 best t= 0
finished: t= 126 loss= 0.17169377207756042 loss val= 0.19486743211746216 best val loss= 0.19020992517471313 best t= 122
finished: t= 252 loss= 0.21795722842216492 loss val= 0.1946682333946228 best val loss= 0.18560203909873962 best t= 130
finished: t= 378 loss= 0.2064885050058365 loss val= 0.19532808661460876 best val loss= 0.1838226616382599 best t= 367
finished: t= 504 loss= 0.18580105900764465 loss val= 0.1910000741481781 best val loss= 0.17913465201854706 best t= 471
finished: t= 630 loss= 0.18896082043647766 loss val= 0.18138930201530457 best val loss= 0.17675703763961792 best t= 610
finished: t= 756 loss= 0.20254294574260712 loss val= 0.18793350458145142 best val loss= 0.17675703763961792 best t= 610
finished: t= 882 loss= 0.1802608221769333 loss val= 0.1896347999572754 best val loss= 0.1747458130121231 best t= 777
finished: t= 1008 loss= 0.18433178961277008 loss val= 0

In [8]:
## MI estimate via flows (MIENF)
from estimators import MIENF

estimator = MIENF(hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

MIENF (K=1), joint learning True 

finished: t= 0 loss= 10100.0126953125 loss val= 10164.837890625 best val loss= 10164.837890625 best t= 0
finished: t= 101 loss= -67.45845031738281 loss val= -65.74169921875 best val loss= -65.74169921875 best t= 101
finished: t= 202 loss= -80.4241714477539 loss val= -78.30210876464844 best val loss= -78.48472595214844 best t= 199
finished: t= 303 loss= -83.72708129882812 loss val= -82.29544830322266 best val loss= -82.91329956054688 best t= 296
finished: t= 404 loss= -87.3494873046875 loss val= -86.1624526977539 best val loss= -86.27616882324219 best t= 403
finished: t= 505 loss= -89.897705078125 loss val= -88.57675170898438 best val loss= -89.02144622802734 best t= 495
finished: t= 606 loss= -93.27511596679688 loss val= -90.2864990234375 best val loss= -90.53706359863281 best t= 596
finished: t= 707 loss= -93.57966613769531 loss val= -90.76760864257812 best val loss= -91.41366577148438 best t= 686
finished: t= 808 loss= -94.97583770751953 loss val= -